# Exclusive attempts and supervised worker processes

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
"""Explicit fresh-run launcher; supervises training and fresh-process evaluation."""
from pathlib import Path
import argparse,json,os,signal,socket,subprocess,sys,time
from contract import ROOT,PYTHON,resolve,reserve_attempt,read_json,write_json,sha,worker_command

def execute_stage(command,stage,timeout,worker):
    started=time.monotonic();timed_out=False
    with (stage.parent/(stage.name+'-process.log')).open('x') as log:
        process=subprocess.Popen(command,stdout=log,stderr=subprocess.STDOUT,start_new_session=True,
            cwd=ROOT, env={**{k:v for k,v in os.environ.items() if k not in ('PYTHONPATH','PYTHONHOME')},'PYTHONDONTWRITEBYTECODE':'1'})
        try:
            while process.poll() is None:
                if time.monotonic()-started>timeout:
                    timed_out=True;os.killpg(process.pid,signal.SIGTERM)
                    try:process.wait(timeout=25)
                    except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL)
                    break
                time.sleep(.2)
            exit_code=process.wait()
        except BaseException:
            if process.poll() is None:os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=25)
            except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()
            raise
    data=read_json(stage/'result.json') if (stage/'result.json').exists() else {}
    pid_data=read_json(stage/'process.json') if (stage/'process.json').exists() else {}
    pid=pid_data.get('unity_pid');alive=False
    if pid:
        try:os.kill(pid,0);alive=True
        except ProcessLookupError:pass
    if alive:
        # Worker group is ours; never kill an unrelated process or clear arbitrary ports.
        try:os.killpg(process.pid,signal.SIGTERM)
        except ProcessLookupError:pass
    with socket.socket() as sock:
        sock.settimeout(.5);no_connection=sock.connect_ex(('127.0.0.1',5005+worker))!=0
    with socket.socket() as sock:
        sock.setsockopt(socket.SOL_SOCKET,socket.SO_REUSEADDR,1)
        try:sock.bind(('127.0.0.1',5005+worker));reusable=True
        except OSError:reusable=False
    result={'stage':stage.name,'worker_pid':process.pid,'command':command,'exit_code':exit_code,'timed_out':timed_out,
        'unity_pid':pid,'unity_alive_after_worker':alive,'no_tcp_connection_after':no_connection,'port_reusable':reusable,
        'stage_status':data.get('status'),'wall_seconds':time.monotonic()-started}
    if exit_code or timed_out or alive or not no_connection or not reusable or data.get('status')!='PASS':
        write_json(stage.parent/(stage.name+'-supervision.json'),result);raise RuntimeError('Stage failed; no automatic retry: '+stage.name)
    write_json(stage.parent/(stage.name+'-supervision.json'),result);return result


class ExperimentSupervisor:
    reserve_attempt=staticmethod(reserve_attempt)
    launch_worker=staticmethod(execute_stage)
    def run(self,config_path,attempt='attempt-01',worker=176,restart_of=None):
        cfg=resolve(config_path,attempt,worker)
        target=reserve_attempt(ROOT/'outputs/runs'/cfg['run_id'],attempt,restart_of)
        cfg['restart_of']=restart_of; resolved=target/'resolved_config.json';write_json(resolved,cfg)
        stages=[];started=time.monotonic();error=None;status='FAIL'
        try:
            if cfg['condition']!='RANDOM':
                train=target/'train'
                stages.append(execute_stage(worker_command('policy_training',['--mode','train','--resolved',resolved,'--stage-dir',train,'--simulator-seed',cfg['simulator_initialization_seed']]),train,cfg['training_timeout_seconds'],worker))
                checkpoint=train/f"checkpoint_{cfg['training_decisions']}.zip"
            for seed in cfg['evaluation']['requested_simulator_initialization_seeds']:
                stage=target/f'evaluate-{seed}'
                args=['--resolved',resolved,'--stage-dir',stage,'--simulator-seed',seed]
                if cfg['condition']=='RANDOM':
                    module='random_evaluation';args+=['--action-seed',seed+1000]
                else:
                    module='policy_training';args+=['--mode','evaluate','--checkpoint',checkpoint,'--training-dir',train]
                stages.append(execute_stage(worker_command(module,args),stage,cfg['evaluation_timeout_seconds'],worker))
            status='PASS'
        except BaseException as exc:error=repr(exc)
        result={'status':status,'run_id':cfg['run_id'],'attempt_id':attempt,'stages':stages,'exception':error,'wall_seconds':time.monotonic()-started,'automatic_retries':0,'resolved_config_sha256':sha(resolved)}
        write_json(target/'result.json',result)
        if status=='PASS':
            marker={'run_id':cfg['run_id'],'attempt_id':attempt,'result_sha256':sha(target/'result.json')}
            write_json(target/'COMPLETE.json',marker);write_json(target.parent/'COMPLETE.json',marker)
        else:raise RuntimeError(result)
        return result
print('Exclusive attempts and supervised worker processes definitions/execution completed.')


Frozen runtime contract definitions/execution completed.
Exclusive attempts and supervised worker processes definitions/execution completed.
